In [55]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [56]:
pip install pandera

Note: you may need to restart the kernel to use updated packages.


In [57]:
import pandas as pd
import pandera.pandas as pa
from pandera.pandas import Column, Check, DataFrameSchema

In [58]:
file_path = "external_data/32180DS0003_2001-25.xlsx"

# Load everything as string first - no automatic type inference at all
table1 = pd.read_excel(
    file_path,
    sheet_name="Table 1",
    header=5,
    skipfooter=4,
    dtype=str
)

table1.head()

,S/T code,S/T name,GCCSA code,GCCSA name,SA4 code,SA4 name,SA3 code,SA3 name,SA2 code,SA2 name,no.,no..1,no..2,no..3,no..4,no..5,no..6,no..7,no..8,no..9,no..10,no..11,no..12,no..13,no..14,no..15,no..16,no..17,no..18,no..19,no..20,no..21,no..22,no..23,no..24
0,1,New South Wales,1RNSW,Rest of NSW,101,Capital Region,10102,Queanbeyan,101021007,Braidwood,2760,2811,2835,2844,2847,2965,3102,3181,3308,3429,3458,3590,3685,3762,3849,3950,4041,4145,4218,4282,4332,4368,4389,4480,4494
1,1,New South Wales,1RNSW,Rest of NSW,101,Capital Region,10102,Queanbeyan,101021008,Karabar,9129,9199,9263,9277,9209,9212,9033,8994,9030,9101,9092,9012,8848,8731,8603,8531,8530,8516,8500,8535,8548,8527,8470,8432,8409
2,1,New South Wales,1RNSW,Rest of NSW,101,Capital Region,10102,Queanbeyan,101021009,Queanbeyan,9717,9513,9522,9400,9595,9682,9793,10074,10288,10584,10715,10870,11050,11199,11213,11230,11362,11460,11468,11460,11375,11378,11391,11344,11251
3,1,New South Wales,1RNSW,Rest of NSW,101,Capital Region,10102,Queanbeyan,101021010,Queanbeyan - East,3925,4073,4219,4218,4187,4319,4459,4595,4712,4813,4951,4971,4983,4967,4961,4970,5016,5079,5126,5089,5097,5101,5099,5133,5111
4,1,New South Wales,1RNSW,Rest of NSW,101,Capital Region,10102,Queanbeyan,101021012,Queanbeyan West - Jerrabomberra,9425,10257,11085,11549,12046,12358,12622,12798,12988,13146,13223,13204,13210,13193,13164,13150,13090,13022,12955,12821,12748,12790,12861,12872,12735


In [59]:
# Drop the two problem rows outright:
# - Index 2453: Norfolk Island (S/T code = 9, "Other Territories") - not part of 
#   Victoria, and has genuine historical data gaps (".." for 2001-2015) since it 
#   only joined mainland ERP reporting around 2016. 
#  Irrelevant to a Victoria-only project.
# - Index 2454: "Total Australia" - this is a national aggregate/summary row, 
#  not  an actual SA2 area. It was never real geographic data to begin with, so
#  it should never have been validated as an SA2 row in the first place.
table1 = table1.drop(index=[2453, 2454])

print(table1.shape)  # should be 2 rows fewer than before

(2453, 35)


In [61]:
old_cols = ['no.'] + [f'no..{i}' for i in range(1, 25)]
new_cols = list(range(2001, 2026))
rename_map = dict(zip(old_cols, new_cols))

table1 = table1.rename(columns=rename_map)

In [62]:
schema_dict = {
    # --- Geography codes: stay as strings, fixed-length per ASGS standard ---
    'S/T code':    Column(str, Check.str_length(1, 1), nullable=False),
    'GCCSA code':  Column(str, nullable=False),  # alphanumeric e.g. '1RNSW' - no fixed length check
    'SA4 code':    Column(str, Check.str_length(3, 3), nullable=False),
    'SA3 code':    Column(str, Check.str_length(5, 5), nullable=False),
    'SA2 code':    Column(str, Check.str_length(9, 9), nullable=False),

    # --- Geography names: non-null strings, no length constraint ---
    'S/T name':    Column(str, nullable=False),
    'GCCSA name':  Column(str, nullable=False),
    'SA4 name':    Column(str, nullable=False),
    'SA3 name':    Column(str, nullable=False),
    'SA2 name':    Column(str, nullable=False),
}

In [63]:
# --- Year columns 2001-2025: currently str, coerce to Int64, must be positive ---
for year in range(2001, 2026):
    schema_dict[year] = Column(
        'Int64',
        nullable=False,
        coerce=True
    )

table1_schema = DataFrameSchema(schema_dict, strict=True)

In [64]:
try:
    validated = table1_schema.validate(table1, lazy=True)
    print("All rows passed schema validation.")
    table1 = validated  # replace with the coerced version (years now Int64)
except pa.errors.SchemaErrors as err:
    print(f"{len(err.failure_cases)} schema violations found.\n")
    print(err.failure_cases)

    failing_indices = err.failure_cases['index'].dropna().unique()
    print(f"\nFull rows that failed validation ({len(failing_indices)} rows):")
    print(table1.loc[failing_indices])

All rows passed schema validation.
